In [4]:
import pandas as pd
import geopandas as gpd
import numpy as np
import os
from shapely.geometry import Point, LineString, Polygon
from datetime import datetime

In [5]:
myworkingdirectory = r"C:\Users\Sophie.Kaye\Desktop\NPS Crash Data"
os.chdir(myworkingdirectory)

In [6]:
imars_crash_new = pd.read_excel("./IMARS/All MVCs April 2024 to September 2024 NO PII.xlsx", sheet_name = "Address")
imars_passenger_new = pd.read_excel("./IMARS/All MVCs April 2024 to September 2024 NO PII.xlsx", sheet_name = "Person")
imars_vehicle_new = pd.read_excel("./IMARS/All MVCs April 2024 to September 2024 NO PII.xlsx", sheet_name = "Vehicle")
imars_crash_details_new = pd.read_excel("./IMARS/All MVCs April 2024 to September 2024 NO PII.xlsx", sheet_name = "Incidnts")
imars_crash_USPP = pd.read_excel("./IMARS/USPP_GOccIvPA Command Address 2018-2024_noPII.xlsx")
imars_passenger_USPP = pd.read_excel("./IMARS/USPP_MVCOccGPersonReport Command 2018-2024_noPII.xlsx")
imars_vehicle_USPP = pd.read_excel("./IMARS/USPP_MVCOccVehicleReport 2018-2024 _noPII.xlsx")
imars_crash_details_USPP = pd.read_excel("./IMARS/USPP_MVCOccReportCommand 2018-2024_noPII.xlsx")
imars_crash_PR = pd.read_excel("./IMARS/GOccIvPA Command Address 1-1-2022 to 3-31-2024 _noPII.xlsx")
imars_passenger_PR = pd.read_excel("./IMARS/MVCOccGPersonReport Command 1-1-2022 to 3-31-2024 _noPII.xlsx")
imars_vehicle_PR = pd.read_excel("./IMARS/MVCOccVehicleReport Command 1-1-2022 to 3-31-2024_noPII.xlsx")
imars_crash_details_PR = pd.read_excel("./IMARS/MVCOccReportCommand 1-1-2022 to 3-31-2024_noPII.xlsx")
imars_crash_old = pd.read_excel("./IMARS/IMARS 2012 - 2021.xlsx", sheet_name = "Location")
imars_passenger_old = pd.read_excel("./IMARS/IMARS 2012 - 2021.xlsx", sheet_name = "Person")
imars_vehicle_old = pd.read_excel("./IMARS/IMARS 2012 - 2021.xlsx", sheet_name = "Vehicle")
imars_crash_details_old = pd.read_excel("./IMARS/IMARS 2012 - 2021.xlsx", sheet_name = "Classification")

In [7]:
imars_crash_USPP.columns

Index(['IMARS_Record_No', 'Crash_Date_Time', 'Linked_Address_Classification',
       'City_Town_Park_Location', 'State', 'County', 'Direction',
       'Linked_Street_Number', 'Linked_Common_Name', 'Street_Type',
       'Direction.1', 'NEAR_Distance_to_MI', 'NEAR_Direction_To',
       'NEAR_Direction', 'NEAR_route_street_road_name', 'NEAR_Road_Type',
       'NEAR_Direction.1', 'AT_Intersection_route_street_road_DIRECTION',
       'At_Intersecting_route_street_road_name', 'AT_Road_Type',
       'AT_Direction', 'Mile_Marker', 'Latitude', 'Longitude', 'Region',
       'State_Zone', 'Park', 'Site', 'Place', 'Point',
       'Road_Type_Classification', 'Linked_Address'],
      dtype='object')

In [8]:
imars_crash_PR.columns

Index(['IMARS_Record_No', 'Crash_Date_Time', 'Linked_Address_Classification',
       'City_Town_Park_Location', 'State', 'County', 'Direction',
       'Linked_Street_Number', 'Linked_Common_Name', 'Street_Type',
       'Direction.1', 'NEAR_Distance_to_MI', 'NEAR_Direction_To',
       'NEAR_Direction', 'NEAR_route_street_road_name', 'NEAR_Road_Type',
       'NEAR_Direction.1', 'AT_Intersection_route_street_road_DIRECTION',
       'At_Intersecting_route_street_road_name', 'AT_Road_Type',
       'AT_Direction', 'Mile_Marker', 'Latitude', 'Longitude', 'Region',
       'State_Zone', 'Park', 'Site', 'Place', 'Point',
       'Road_Type_Classification', 'Linked_Address'],
      dtype='object')

In [9]:
imars_crash_USPP.shape, imars_crash_PR.shape

((8704, 32), (5962, 32))

In [10]:
8704+5962

14666

In [11]:
imars_crash = pd.concat([imars_crash_USPP,imars_crash_PR])
imars_crash.shape

(14666, 32)

In [12]:
# no crashes in common between datasets - we can just append the new rows to the existing data
test = pd.merge(imars_crash, imars_crash_new, how='inner', on='IMARS_Record_No')
test.shape[0]

0

In [13]:
imars_crash_new.columns

Index(['IMARS_Record_No', 'Crash_Date_Time', 'Linked_Address_Classification',
       'City_Town_Park_Location', 'State', 'County', 'Direction',
       'Linked_Street_Number', 'Linked_Common_Name', 'Street_Type',
       'Direction_1', 'NEAR_Distance_to_MI', 'NEAR_Direction_To',
       'NEAR_Direction', 'NEAR_route_street_road_name', 'NEAR_Road_Type',
       'NEAR_Direction_2', 'AT_Intersection_route_street_road_DIRECTION',
       'At_Intersecting_route_street_road_name', 'AT_Road_Type',
       'AT_Direction', 'Mile_Marker', 'Latitude', 'Longitude', 'Region',
       'State_Zone', 'Park', 'Site', 'Place', 'Point',
       'Road_Type_Classification', 'Linked_Address'],
      dtype='object')

In [14]:
imars_crash_new = imars_crash_new.rename(columns={'Direction_1':'Direction.1','NEAR_Direction_2':'NEAR_Direction.1'})

In [15]:
imars_crash_new.shape

(2323, 32)

In [16]:
14666+2323

16989

In [17]:
imars_crash = pd.concat([imars_crash, imars_crash_new])
imars_crash.shape

(16989, 32)

In [18]:
# drop rows from old database that are common with new database such that updated reports will prevail in final dataset with no overlap
imars_crash_old_nodupes = imars_crash_old.loc[~imars_crash_old['IMARS_Record_No'].isin(imars_crash['IMARS_Record_No'])] # Note that tilde symbol indicates NOT IN
imars_crash_old_nodupes.shape

(15640, 32)

In [19]:
imars_crash_raw_combined = pd.concat([imars_crash, imars_crash_old_nodupes])
imars_crash_raw_combined.shape

(32629, 32)

In [20]:
imars_crash_raw_combined.shape[0], (imars_crash_old_nodupes.shape[0]+imars_crash.shape[0])

(32629, 32629)

In [21]:
# output raw combined dataset for fill percentage investigation
imars_crash_raw_combined.to_csv('./IMARS/IMARS_crash_yearscombined_raw.csv', index=False)

In [22]:
imars_passenger_USPP.shape, imars_passenger_PR.shape

((17491, 30), (9433, 30))

In [23]:
17491+9433

26924

In [24]:
imars_passenger = pd.concat([imars_passenger_USPP,imars_passenger_PR])
imars_passenger.shape

(26924, 30)

In [25]:
imars_passenger.columns

Index(['IMARS_Record_No', 'Crash_Date_Time', 'ParkUnit', 'Driver_Action',
       'Driver_Condition', 'Driver_Distraction', 'Suspect_Alcohol',
       'Alcohol_Test', 'Alcohol_Test_Result_1', 'Alcohol_Test_Result_2',
       'Suspect_Drugs', 'Drug_Test', 'Violations_Issued', 'Seat_Position',
       'Injury_Severity', 'Air_Bag_Deployed', 'Ejection',
       'Injury_Transported_By', 'Safety_Equipment_Used', 'Vehicle_number',
       'Injured_transported_by', 'Non_motorist_safety_equipment',
       'Non_motorist_action_circumstance_prior_to_crash',
       'Non_motorist_action_circumstance_at_time_of_crash',
       'Non_motorist_condition_at_time_of_crash', 'Non_motorist_distraction',
       'Non_motorist_location_at_time_of_crash', 'Pedestrian_Type',
       'Pedestrian_Type_Detail', 'Involvement'],
      dtype='object')

In [26]:
imars_passenger = imars_passenger.drop(columns='ParkUnit')
imars_passenger.columns

Index(['IMARS_Record_No', 'Crash_Date_Time', 'Driver_Action',
       'Driver_Condition', 'Driver_Distraction', 'Suspect_Alcohol',
       'Alcohol_Test', 'Alcohol_Test_Result_1', 'Alcohol_Test_Result_2',
       'Suspect_Drugs', 'Drug_Test', 'Violations_Issued', 'Seat_Position',
       'Injury_Severity', 'Air_Bag_Deployed', 'Ejection',
       'Injury_Transported_By', 'Safety_Equipment_Used', 'Vehicle_number',
       'Injured_transported_by', 'Non_motorist_safety_equipment',
       'Non_motorist_action_circumstance_prior_to_crash',
       'Non_motorist_action_circumstance_at_time_of_crash',
       'Non_motorist_condition_at_time_of_crash', 'Non_motorist_distraction',
       'Non_motorist_location_at_time_of_crash', 'Pedestrian_Type',
       'Pedestrian_Type_Detail', 'Involvement'],
      dtype='object')

In [27]:
imars_passenger_old.shape

(27932, 29)

In [28]:
# drop rows from old database that are common with new database such that updated reports will prevail in final dataset with no overlap
imars_passenger_old_nodupes = imars_passenger_old.loc[~imars_passenger_old['IMARS_Record_No'].isin(imars_passenger['IMARS_Record_No'])] # Note that tilde symbol indicates NOT IN
imars_passenger_old_nodupes.shape

(23439, 29)

In [29]:
imars_passenger_new.shape

(4232, 30)

In [30]:
imars_passenger_new.columns

Index(['IMARS_Record_No', 'Crash_Date_Time', 'ParkUnit', 'Driver_Action',
       'Driver_Condition', 'Driver_Distraction', 'Suspect_Alcohol',
       'Alcohol_Test', 'Alcohol_Test_Result_1', 'Alcohol_Test_Result_2',
       'Suspect_Drugs', 'Drug_Test', 'Violations_Issued', 'Seat_Position',
       'Injury_Severity', 'Air_Bag_Deployed', 'Ejection',
       'Injury_Transported_By', 'Safety_Equipment_Used', 'Vehicle_number',
       'Injured_transported_by', 'Non_motorist_safety_equipment',
       'Non_motorist_action_circumstance_prior_to_crash',
       'Non_motorist_action_circumstance_at_time_of_crash',
       'Non_motorist_condition_at_time_of_crash', 'Non_motorist_distraction',
       'Non_motorist_location_at_time_of_crash', 'Pedestrian_Type',
       'Pedestrian_Type_Detail', 'Involvement'],
      dtype='object')

In [31]:
imars_passenger_new = imars_passenger_new.drop(columns='ParkUnit')
imars_passenger_new.columns

Index(['IMARS_Record_No', 'Crash_Date_Time', 'Driver_Action',
       'Driver_Condition', 'Driver_Distraction', 'Suspect_Alcohol',
       'Alcohol_Test', 'Alcohol_Test_Result_1', 'Alcohol_Test_Result_2',
       'Suspect_Drugs', 'Drug_Test', 'Violations_Issued', 'Seat_Position',
       'Injury_Severity', 'Air_Bag_Deployed', 'Ejection',
       'Injury_Transported_By', 'Safety_Equipment_Used', 'Vehicle_number',
       'Injured_transported_by', 'Non_motorist_safety_equipment',
       'Non_motorist_action_circumstance_prior_to_crash',
       'Non_motorist_action_circumstance_at_time_of_crash',
       'Non_motorist_condition_at_time_of_crash', 'Non_motorist_distraction',
       'Non_motorist_location_at_time_of_crash', 'Pedestrian_Type',
       'Pedestrian_Type_Detail', 'Involvement'],
      dtype='object')

In [32]:
imars_passenger_raw_combined = pd.concat([imars_passenger_new, imars_passenger, imars_passenger_old_nodupes])
imars_passenger_raw_combined.shape

(54595, 29)

In [33]:
imars_passenger_raw_combined.shape[0], (imars_passenger_old_nodupes.shape[0]+imars_passenger.shape[0]+imars_passenger_new.shape[0])

(54595, 54595)

In [34]:
# output raw combined dataset for fill percentage investigation
imars_passenger_raw_combined.to_csv('./IMARS/IMARS_passenger_yearscombined_raw.csv', index=False)

In [35]:
imars_vehicle_USPP.shape, imars_vehicle_PR.shape

((14946, 53), (8068, 53))

In [36]:
14946+8068

23014

In [37]:
imars_vehicle = pd.concat([imars_vehicle_USPP,imars_vehicle_PR])
imars_vehicle.shape

(23014, 53)

In [38]:
imars_vehicle.columns

Index(['IMARS_Record_No', 'Crash_Date_Time', 'ParkUnit', 'Vehicle_Number',
       'Number_of_Occupants', 'Vehicle_Towed', 'Insurance_verified',
       'Initial_Impact_Point', 'Most_Damaged_Area', 'Extent_of_Damage',
       'Direction_of_Travel_Prior_to_Crash', 'Posted_Speed',
       'First_Event_Type', 'First_Event', 'Second_Event_Type', 'Second_Event',
       'Third_Event_Type', 'Third_Event', 'Fourth_Event_Type', 'Fourth_Event',
       'Motor_Vehicle_Unit_Type', 'Most_Harmful_Event', 'Vehicle_Owner',
       'Vehicle_Type', 'Non_Commercial_Trailer_Style', 'Emergency_Vehicle_Use',
       'Emergency_Equipment_Activated', 'Special_Function_of_MV_in_Transport',
       'Motor_Vehicle_Contributing_Circumstance',
       'Vehicle_Maneuver_Action_Prior_to_Crash', 'Road_Surface', 'Grade',
       'Roadway_Alignment', 'Total_Number_of_Lanes', 'Traffic_Control',
       'Traffic_Control_Working_Properly', 'Roadway_Description',
       'Commercial_Non_Commercial', 'Number_of_Axles', 'Gross_Vehicle_W

In [39]:
imars_vehicle = imars_vehicle.drop(columns=['ParkUnit','Most_Harmful_Event'])

In [40]:
imars_vehicle_old.columns

Index(['IMARS_Record_No', 'Crash_Date_Time', 'Vehicle_Number',
       'Number_of_Occupants', 'Vehicle_Towed', 'Insurance_verified',
       'Initial_Impact_Point', 'Most_Damaged_Area', 'Extent_of_Damage',
       'Direction_of_Travel_Prior_to_Crash', 'Posted_Speed',
       'First_Event_Type', 'First_Event', 'Second_Event_Type', 'Second_Event',
       'Third_Event_Type', 'Third_Event', 'Fourth_Event_Type', 'Fourth_Event',
       'Motor_Vehicle_Unit_Type', 'Vehicle_Owner', 'Vehicle_Type',
       'Non_Commercial_Trailer_Style', 'Emergency_Vehicle_Use',
       'Emergency_Equipment_Activated', 'Special_Function_of_MV_in_Transport',
       'Motor_Vehicle_Contributing_Circumstance',
       'Vehicle_Maneuver_Action_Prior_to_Crash', 'Road_Surface', 'Grade',
       'Roadway_Alignment', 'Total_Number_of_Lanes', 'Traffic_Control',
       'Traffic_Control_Working_Properly', 'Roadway_Description',
       'Commercial_Non_Commercial', 'Number_of_Axles', 'Gross_Vehicle_Weight',
       'Combination_GVW', 

In [41]:
imars_vehicle_new.columns

Index(['IMARS_Record_No', 'Crash_Date_Time', 'ParkUnit', 'Vehicle_Number',
       'Number_of_Occupants', 'Vehicle_Towed', 'Insurance_verified',
       'Initial_Impact_Point', 'Most_Damaged_Area', 'Extent_of_Damage',
       'Direction_of_Travel_Prior_to_Crash', 'Posted_Speed',
       'First_Event_Type', 'First_Event', 'Second_Event_Type', 'Second_Event',
       'Third_Event_Type', 'Third_Event', 'Fourth_Event_Type', 'Fourth_Event',
       'Motor_Vehicle_Unit_Type', 'Most_Harmful_Event', 'Vehicle_Owner',
       'Vehicle_Type', 'Non_Commercial_Trailer_Style', 'Emergency_Vehicle_Use',
       'Emergency_Equipment_Activated', 'Special_Function_of_MV_in_Transport',
       'Motor_Vehicle_Contributing_Circumstance',
       'Vehicle_Maneuver_Action_Prior_to_Crash', 'Road_Surface', 'Grade',
       'Roadway_Alignment', 'Total_Number_of_Lanes', 'Traffic_Control',
       'Traffic_Control_Working_Properly', 'Roadway_Description',
       'Commercial_Non_Commercial', 'Number_of_Axles', 'Gross_Vehicle_W

In [42]:
imars_vehicle_new = imars_vehicle_new.drop(columns=['ParkUnit','Most_Harmful_Event'])

In [43]:
# drop rows from old database that are common with new database such that updated reports will prevail in final dataset with no overlap
imars_vehicle_old_nodupes = imars_vehicle_old.loc[~imars_vehicle_old['IMARS_Record_No'].isin(imars_vehicle['IMARS_Record_No'])] # Note that tilde symbol indicates NOT IN
imars_vehicle_old_nodupes.shape

(19745, 51)

In [44]:
imars_vehicle_raw_combined = pd.concat([imars_vehicle_new, imars_vehicle, imars_vehicle_old_nodupes])
imars_vehicle_raw_combined.shape[0], (imars_vehicle_old_nodupes.shape[0]+imars_vehicle.shape[0]+imars_vehicle_new.shape[0])

(46345, 46345)

In [45]:
# output raw combined dataset for fill percentage investigation
imars_vehicle_raw_combined.to_csv('./IMARS/IMARS_vehicle_yearscombined_raw.csv', index=False)

In [46]:
imars_crash_details_USPP.shape, imars_crash_details_PR.shape

((8042, 27), (5782, 27))

In [47]:
8042+5782

13824

In [48]:
imars_crash_details = pd.concat([imars_crash_details_USPP,imars_crash_details_PR])
imars_crash_details.shape

(13824, 27)

In [49]:
imars_crash_details.columns

Index(['IMARS_Record_No', 'ParkUnit', 'Crash_Date_Time', 'Crash_Date',
       'DateCreated', 'Number_of_Vehicles_Involved', 'Injury_or_Fatal_Crash',
       'Investigated_at_Scene', 'Hit_and_Run',
       'Non_Motor_Vehicl_Property_Damage', 'Amount_of_Property_Damage',
       'First_Harmful_Event_Type', 'First_Harmful_Event',
       'Location_of_First_Harmful_Event', 'Weather', 'Roadway_Condition',
       'Lighting', 'School_Bus_related', 'AS_Road_Circumstance',
       'Environmental_Contributing_Circumstances', 'Work_Zone_Related',
       'Work_Zone_Workers_Present', 'Work_Zone_Location',
       'Law_Enforcement_Present_at_Work_Zone', 'Relation_to_Junction',
       'Type_of_Intersection', 'Manner_of_Collision'],
      dtype='object')

In [50]:
imars_crash_details = imars_crash_details.drop(columns=['ParkUnit','Crash_Date','DateCreated'])
imars_crash_details.shape

(13824, 24)

In [51]:
# drop rows from old database that are common with new database such that updated reports will prevail in final dataset with no overlap
imars_crash_details_old_nodupes = imars_crash_details_old.loc[~imars_crash_details_old['IMARS_Record_No'].isin(imars_crash_details['IMARS_Record_No'])] # Note that tilde symbol indicates NOT IN
imars_crash_details_old_nodupes.shape

(14255, 24)

In [52]:
imars_crash_details_old.columns

Index(['IMARS_Record_No', 'Crash_Date_Time', 'Number_of_Vehicles_Involved',
       'Injury_or_Fatal_Crash', 'Investigated_at_Scene', 'Hit_and_Run',
       'Non_Motor_Vehicl_Property_Damage', 'Amount_of_Property_Damage',
       'First_Harmful_Event_Type', 'First_Harmful_Event',
       'Location_of_First_Harmful_Event', 'Weather', 'Roadway_Condition',
       'Lighting', 'School_Bus_related', 'AS_Road_Circumstance',
       'Environmental_Contributing_Circumstances', 'Work_Zone_Related',
       'Work_Zone_Workers_Present', 'Work_Zone_Location',
       'Law_Enforcement_Present_at_Work_Zone', 'Relation_to_Junction',
       'Type_of_Intersection', 'Manner_of_Collision'],
      dtype='object')

In [53]:
imars_crash_details_new.shape

(2341, 26)

In [54]:
imars_crash_details_new.columns

Index(['IMARS_Record_No', 'ParkUnit', 'Crash_Date_Time', 'DateCreated',
       'Number_of_Vehicles_Involved', 'Injury_or_Fatal_Crash',
       'Investigated_at_Scene', 'Hit_and_Run',
       'Non_Motor_Vehicl_Property_Damage', 'Amount_of_Property_Damage',
       'First_Harmful_Event_Type', 'First_Harmful_Event',
       'Location_of_First_Harmful_Event', 'Weather', 'Roadway_Condition',
       'Lighting', 'School_Bus_related', 'AS_Road_Circumstance',
       'Environmental_Contributing_Circumstances', 'Work_Zone_Related',
       'Work_Zone_Workers_Present', 'Work_Zone_Location',
       'Law_Enforcement_Present_at_Work_Zone', 'Relation_to_Junction',
       'Type_of_Intersection', 'Manner_of_Collision'],
      dtype='object')

In [55]:
imars_crash_details_new = imars_crash_details_new.drop(columns=['ParkUnit','DateCreated'])
imars_crash_details_new.shape

(2341, 24)

In [56]:
imars_crash_details_raw_combined = pd.concat([imars_crash_details_new, imars_crash_details, imars_crash_details_old_nodupes])
imars_crash_details_raw_combined.shape[0], (imars_crash_details_old_nodupes.shape[0]+imars_crash_details.shape[0]+imars_crash_details_new.shape[0])

(30420, 30420)

In [57]:
# output raw combined dataset for fill percentage investigation
imars_crash_details_raw_combined.to_csv('./IMARS/IMARS_crash_details_yearscombined_raw.csv', index=False)

In [58]:
imars_crash_details_raw_combined.shape, imars_crash_raw_combined.shape, imars_passenger_raw_combined.shape, imars_vehicle_raw_combined.shape

((30420, 24), (32629, 32), (54595, 29), (46345, 51))

# General Data Cleaning
Note that there are crashes with randomly missing park, region, and/or roadway

### Remove crashes missing record numbers

In [59]:
imars_crash = imars_crash_raw_combined.dropna(subset=['IMARS_Record_No'])
imars_passenger = imars_passenger_raw_combined.dropna(subset=['IMARS_Record_No'])
imars_vehicle = imars_vehicle_raw_combined.dropna(subset=['IMARS_Record_No'])
imars_crash_details = imars_crash_details_raw_combined.dropna(subset=['IMARS_Record_No'])

### Remove duplicate reports

In [60]:
imars_crash = imars_crash.drop_duplicates(subset=['IMARS_Record_No'])
imars_crash_details = imars_crash_details.drop_duplicates(subset=['IMARS_Record_No'])
# Cannot remove duplicates from vehicle and passenger tables because they are unique records for each person/car involved in crash 

### Revert flipped lat/long

In [61]:
need_revert = imars_crash['Latitude'].abs() > 70
imars_crash.loc[need_revert, ['Latitude', 'Longitude']] = (imars_crash.loc[need_revert, ['Longitude', 'Latitude']].values)

### Adjust coordinate signs to correct hemisphere

In [62]:
imars_crash['Latitude'].loc[imars_crash['Latitude'] < 0] = imars_crash['Latitude'].loc[imars_crash['Latitude'] < 0]*(-1)
imars_crash['Longitude'].loc[imars_crash['Longitude'] > 0] = imars_crash['Longitude'].loc[imars_crash['Longitude'] > 0]*(-1)

In [63]:
imars_crash['Longitude'].loc[imars_crash['Longitude'].abs() > 180] = imars_crash['Longitude'].loc[imars_crash['Longitude'].abs() > 180]/10
imars_crash['Latitude'].loc[imars_crash['Latitude'].abs() > 80] = imars_crash['Latitude'].loc[imars_crash['Latitude'].abs() > 80]/10

C:\Users\Sophie.Kaye\AppData\Local\Temp\ipykernel_15636\1536543634.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  imars_crash['Longitude'].loc[imars_crash['Longitude'].abs() > 180] = imars_crash['Longitude'].loc[imars_crash['Longitude'].abs() > 180]/10
C:\Users\Sophie.Kaye\AppData\Local\Temp\ipykernel_15636\1536543634.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  imars_crash['Latitude'].loc[imars_crash['Latitude'].abs() > 80] = imars_crash['Latitude'].loc[imars_crash['Latitude'].abs() > 80]/10


# Add Parks to Crash Table

**Note:** IMARS does not have park units with each crash, so these will have to be added using the Latitude and Longitude fields in the imars_crash dataset. 

In [64]:
imars_crash_parks = imars_crash.dropna(subset=['Park'])
imars_crash_parks.shape[0]

25829

In [65]:
imars_crash.loc[imars_crash['Park'].isnull()].shape[0]
# 3281 crash entries have no park identification

3281

In [66]:
imars_crash_coords = imars_crash.loc[imars_crash['Park'].isnull()].dropna(subset=['Latitude','Longitude'])
imars_crash_coords.shape[0]
# of the 3281 crash entries without park identification, 691 have lat/long coordinates from which park can be assigned using the shapefile

691

In [67]:
# the sum of crashes with pre-identified parks and crashes that can be assigned a park using lat/long coordinates 
# results in the following total of possible usable crash entries after the spatial join:
25829+691

26520

In [68]:
imars_crash_noparks = imars_crash.loc[imars_crash['Park'].isnull()]
imars_crash_noparks_nocoords = imars_crash_noparks.loc[imars_crash_noparks['Latitude'].isnull()]
# 2590 of the crash entries have no park identification or lat/long coordinates
3281-691

2590

In [69]:
mask = imars_crash_noparks_nocoords.loc[:,['Linked_Common_Name','Linked_Address','NEAR_route_street_road_name',
                                           'At_Intersecting_route_street_road_name']].notnull()
imars_noparks_nocoords_someinfo = imars_crash_noparks_nocoords.loc[mask.any(axis=1)]
imars_noparks_nocoords_someinfo.shape[0]
# of the 2590 crash entries without park identification or lat/long coordinates, 148 have other identifiable information 
# (e.g., roadway name) which could be used to identify the park name

148

In [70]:
# print file containing only this subset of crashes for manual park identification
imars_noparks_nocoords_someinfo.to_csv("./IMARS_noparks_nocoords_someinfo_2024.csv",index=False)

## Spatial join to assign Park names

In [71]:
imars_crash_coords_geo=gpd.GeoDataFrame(imars_crash_coords, geometry=gpd.points_from_xy(imars_crash_coords.Longitude, imars_crash_coords.Latitude))
filename = "./shapefiles/NPS_-_Land_Resources_Division_Boundary_and_Tract_Data_Service.geojson"
file = open(filename)
parks = gpd.read_file(file)

imars_crash_coords_geo.crs = "EPSG:4326"
parks = parks.to_crs(epsg=4326)
parks['geometry']=parks['geometry'].buffer(0.01)

imars_crash_coords_geo_withparknames=gpd.sjoin(imars_crash_coords_geo,parks,how="left", predicate='intersects')
#imars_crash_coords_geo_withparknames.to_csv("original_lastidenticalstep.csv", index=False)

imars_crash_coords_geo_withparknames2 = imars_crash_coords_geo_withparknames.drop_duplicates(subset=['IMARS_Record_No'])
imars_crash_coords_geo_withparknames2.shape[0]
# all 691 crash entries were processed in the spatial join

C:\Users\Sophie.Kaye\AppData\Local\Temp\ipykernel_15636\732178193.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  parks['geometry']=parks['geometry'].buffer(0.01)


691

In [72]:
imars_crash_withparknames = pd.DataFrame(imars_crash_coords_geo_withparknames2.drop(columns='geometry'))
imars_crash_withparknames['Park']= imars_crash_withparknames['UNIT_CODE']
#imars_crash_withparknames.to_csv("original.csv", index=False)
imars_crash_withparknames_NoDupsorNulls = imars_crash_withparknames.dropna(subset=['Park'])
imars_crash_withparknames_NoDupsorNulls.shape[0]
# of the 691 crash entries without parks identified, 588 now have parks assigned from spatial join

588

In [73]:
# so we expect the total ("expanded") dataset to be:
25829+588

26417

In [137]:
imars_crash_expanded = pd.concat([imars_crash_parks,imars_crash_withparknames_NoDupsorNulls])
imars_crash_expanded.shape

(26417, 52)

In [138]:
# check to make sure all entries in combined dataset contain park assignments 
imars_crash_expanded = imars_crash_expanded.dropna(subset=['Park'])
imars_crash_expanded.shape[0]

26417

In [139]:
# check to make sure no duplicate crashes in combined dataset
imars_crash_expanded = imars_crash_expanded.drop_duplicates(subset=['IMARS_Record_No'])
imars_crash_expanded.shape[0]

26417

## Add in crashes with Park names manually identified via unmistakable roadway names

In [140]:
# open file I populated in which park names were manually added to crash data containing identifiable info (e.g., road name)
imars_new_parknames = pd.read_csv("./IMARS/IMARS_noparks_nocoords_someinfo_2024_parksfilled.csv")

# remove any entries for which park names were not able to be identified
imars_new_parknames = imars_new_parknames.dropna(subset=['Park'])
imars_new_parknames.shape[0]

101

In [141]:
# 101 of 148 crashes were able to have park names manually added using road names and should be combined with crash data 
# containing park names from original data and spatial assignment for a total of crashes in the final dataset:
26417+101

26518

In [142]:
imars_crash_expanded = pd.concat([imars_crash_expanded,imars_new_parknames])
imars_crash_expanded.shape

(26518, 52)

In [143]:
# check to make sure there are no duplicates in final dataset
imars_crash_expanded = imars_crash_expanded.drop_duplicates(subset=['IMARS_Record_No'])
imars_crash_expanded.shape

(26518, 52)

## Create new Region column
imars_crash database currently contains two columns with region information, although neither one is fully populated. This section will create and populate a new column re-assigning region name to every crash based on park name using a lookup table

In [144]:
imars_crash_expanded.columns

Index(['IMARS_Record_No', 'Crash_Date_Time', 'Linked_Address_Classification',
       'City_Town_Park_Location', 'State', 'County', 'Direction',
       'Linked_Street_Number', 'Linked_Common_Name', 'Street_Type',
       'Direction.1', 'NEAR_Distance_to_MI', 'NEAR_Direction_To',
       'NEAR_Direction', 'NEAR_route_street_road_name', 'NEAR_Road_Type',
       'NEAR_Direction.1', 'AT_Intersection_route_street_road_DIRECTION',
       'At_Intersecting_route_street_road_name', 'AT_Road_Type',
       'AT_Direction', 'Mile_Marker', 'Latitude', 'Longitude', 'Region',
       'State_Zone', 'Park', 'Site', 'Place', 'Point',
       'Road_Type_Classification', 'Linked_Address', 'index_right', 'OBJECTID',
       'UNIT_CODE', 'GIS_Notes', 'UNIT_NAME', 'DATE_EDIT', 'STATE', 'REGION',
       'GNIS_ID', 'UNIT_TYPE', 'CREATED_BY', 'METADATA', 'PARKNAME',
       'CreationDate', 'Creator', 'EditDate', 'Editor', 'GlobalID',
       'Shape__Area', 'Shape__Length'],
      dtype='object')

In [145]:
# load lookup table
park_info = pd.read_csv("./Reference Data/Park_Info_Table.csv")
park_info = park_info.rename(columns={'UNIT_CODE':'Park','REGION':'RGN'})
park_info.columns

Index(['OBJECTID', 'Park', 'GIS_Notes', 'UNIT_NAME', 'DATE_EDIT', 'STATE',
       'RGN', 'GNIS_ID', 'UNIT_TYPE', 'CREATED_BY', 'METADATA', 'PARKNAME',
       'CreationDa', 'Creator', 'EditDate', 'Editor', 'Shape__Are',
       'Shape__Len', 'Urban'],
      dtype='object')

In [146]:
# add RGN column from lookup table to IMARS crash database, joining the two datasets based on park name
imars_crash_expanded = pd.merge(imars_crash_expanded, park_info[['RGN','Park']], how='left', on='Park')
# resulting dataframe after join should have one additional column and no additional rows
imars_crash_expanded = imars_crash_expanded.drop_duplicates() 
imars_crash_expanded.shape

(26518, 53)

In [147]:
# check to see if any crashes were not assigned a region
imars_crash_expanded.loc[imars_crash_expanded['RGN'].isnull()].shape[0]

23

In [148]:
imars_crash_expanded['RGN'].value_counts()

NCR    8311
SER    6246
IMR    4716
PWR    4252
NER    2166
MWR     677
AKR     127
Name: RGN, dtype: int64

In [149]:
# remove partially populated pre-existing columns
imars_crash_expanded = imars_crash_expanded.drop(columns = ['Region', 'REGION'])

In [150]:
imars_crash_expanded.shape

(26518, 51)

In [151]:
imars_crash_expanded.to_csv("IMARS2024_Crash_ParkandRegionAdded_Clean.csv", index= False)

## Park name join to assign Urban/Rural designation

In [152]:
# add Urban column from lookup table to IMARS crash database, joining the two datasets based on park name
imars_crash_expanded = pd.merge(imars_crash_expanded, park_info[['Urban','Park']], how='left', on='Park')
# resulting dataframe after join should have one additional column and no additional rows
imars_crash_expanded = imars_crash_expanded.drop_duplicates() 
imars_crash_expanded.shape

(26518, 52)

In [153]:
imars_crash_expanded.head()

,IMARS_Record_No,Crash_Date_Time,Linked_Address_Classification,City_Town_Park_Location,State,County,Direction,Linked_Street_Number,Linked_Common_Name,Street_Type,...,PARKNAME,CreationDate,Creator,EditDate,Editor,GlobalID,Shape__Area,Shape__Length,RGN,Urban
0,PP17080295,20180803 09:44:00:000,Dispatch address,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NCR,urban
1,PP18027209,20180401 15:43:00:000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NCR,urban
2,PP18027231,20180401 16:39:00:000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NCR,urban
3,PP14068901,20210902 15:12:00:000,Dispatch address,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NCR,NaN
4,PP18027070,20180401 04:30:00:000,Dispatch address,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NCR,urban


In [154]:
# check to see if any crashes were not assigned an urban/rural designation
imars_crash_expanded.loc[imars_crash_expanded['Urban'].isnull()].shape[0]

1265

In [155]:
imars_crash_expanded['Urban'].value_counts()

rural       9561
urban       8412
suburban    7280
Name: Urban, dtype: int64

In [156]:
imars_crash_expanded.to_csv("./IMARS/IMARS2024_Crash_Park_Region_Urban_Added_Clean.csv", index= False)

## Crash Table

In [157]:
# reformat crash date, time, and year; separate into individual columns
imars_crash_expanded['Crash_Date_Time']=pd.to_datetime(imars_crash_expanded['Crash_Date_Time'], format='%Y%m%d %H:%M:%S:%f')
imars_crash_expanded['DATE'] = imars_crash_expanded['Crash_Date_Time'].dt.strftime('%Y-%m-%d')
imars_crash_expanded['YEAR'] = imars_crash_expanded['Crash_Date_Time'].dt.strftime('%Y')
imars_crash_expanded['MONTH'] = imars_crash_expanded['Crash_Date_Time'].dt.strftime('%m')
imars_crash_expanded['DOW'] = imars_crash_expanded['Crash_Date_Time'].dt.strftime('%w')
imars_crash_expanded['HOUR'] = imars_crash_expanded['Crash_Date_Time'].dt.strftime('%H')
imars_crash_expanded['TIME'] = imars_crash_expanded['Crash_Date_Time'].dt.strftime('%H:%M')
imars_crash_expanded.head() # looks good!

,IMARS_Record_No,Crash_Date_Time,Linked_Address_Classification,City_Town_Park_Location,State,County,Direction,Linked_Street_Number,Linked_Common_Name,Street_Type,...,Shape__Area,Shape__Length,RGN,Urban,DATE,YEAR,MONTH,DOW,HOUR,TIME
0,PP17080295,2018-08-03 09:44:00,Dispatch address,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NCR,urban,2018-08-03,2018,08,5,09,09:44
1,PP18027209,2018-04-01 15:43:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NCR,urban,2018-04-01,2018,04,0,15,15:43
2,PP18027231,2018-04-01 16:39:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NCR,urban,2018-04-01,2018,04,0,16,16:39
3,PP14068901,2021-09-02 15:12:00,Dispatch address,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NCR,NaN,2021-09-02,2021,09,4,15,15:12
4,PP18027070,2018-04-01 04:30:00,Dispatch address,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NCR,urban,2018-04-01,2018,04,0,04,04:30


In [158]:
imars_crash_expanded['YEAR'].value_counts()

2023    4130
2019    4012
2018    3642
2022    3610
2021    3442
2024    2971
2020    2882
2017     932
2016     875
2014      11
2013       9
2015       2
Name: YEAR, dtype: int64

In [159]:
imars_crash_expanded.columns

Index(['IMARS_Record_No', 'Crash_Date_Time', 'Linked_Address_Classification',
       'City_Town_Park_Location', 'State', 'County', 'Direction',
       'Linked_Street_Number', 'Linked_Common_Name', 'Street_Type',
       'Direction.1', 'NEAR_Distance_to_MI', 'NEAR_Direction_To',
       'NEAR_Direction', 'NEAR_route_street_road_name', 'NEAR_Road_Type',
       'NEAR_Direction.1', 'AT_Intersection_route_street_road_DIRECTION',
       'At_Intersecting_route_street_road_name', 'AT_Road_Type',
       'AT_Direction', 'Mile_Marker', 'Latitude', 'Longitude', 'State_Zone',
       'Park', 'Site', 'Place', 'Point', 'Road_Type_Classification',
       'Linked_Address', 'index_right', 'OBJECTID', 'UNIT_CODE', 'GIS_Notes',
       'UNIT_NAME', 'DATE_EDIT', 'STATE', 'GNIS_ID', 'UNIT_TYPE', 'CREATED_BY',
       'METADATA', 'PARKNAME', 'CreationDate', 'Creator', 'EditDate', 'Editor',
       'GlobalID', 'Shape__Area', 'Shape__Length', 'RGN', 'Urban', 'DATE',
       'YEAR', 'MONTH', 'DOW', 'HOUR', 'TIME'],
  

In [160]:
imars_crash_expanded = imars_crash_expanded.rename(columns={'Latitude':'LATITUDE','Longitude':'LONGITUDE'}) # to match CDS
imars_crash_slim = imars_crash_expanded[['IMARS_Record_No','LATITUDE','LONGITUDE','Park','RGN','YEAR','MONTH','DOW','HOUR','TIME','Urban','Linked_Address']]
imars_crash_slim.shape

(26518, 12)

In [161]:
imars_crash_slim.to_csv("./IMARS/IMARS2024_crash_slim.csv",index=False)

# join crash slim table to other tables for VRP analysis of 2018-2024

In [162]:
imars_crash_slim['YEAR'] = imars_crash_slim['YEAR'].astype(int)

C:\Users\Sophie.Kaye\AppData\Local\Temp\ipykernel_15636\1638941964.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  imars_crash_slim['YEAR'] = imars_crash_slim['YEAR'].astype(int)


In [163]:
imars_crash_slim_filtered = imars_crash_slim.loc[imars_crash_slim['YEAR']>2017]
imars_crash_slim_filtered['YEAR'].value_counts()

2023    4130
2019    4012
2018    3642
2022    3610
2021    3442
2024    2971
2020    2882
Name: YEAR, dtype: int64

In [164]:
imars_crash_slim_filtered.to_csv('./VRP Needs Assessment/Crash Table.csv', index=False)

In [165]:
imars_passenger.shape, imars_crash_slim_filtered.shape

((54594, 29), (24689, 12))

In [166]:
imars_passenger_crash_filtered = pd.merge(imars_passenger, imars_crash_slim_filtered, how="left", on="IMARS_Record_No")
imars_passenger_crash_filtered.shape

(54594, 40)

In [167]:
imars_passenger_crash_filtered.to_csv('./VRP Needs Assessment/Passenger Table.csv', index=False)

In [168]:
imars_vehicle.shape, imars_crash_slim_filtered.shape

((46344, 51), (24689, 12))

In [169]:
imars_vehicle_crash_filtered = pd.merge(imars_vehicle, imars_crash_slim_filtered, how="left", on="IMARS_Record_No")
imars_vehicle_crash_filtered.shape

(46344, 62)

In [170]:
imars_vehicle_crash_filtered.to_csv('./VRP Needs Assessment/Vehicle Table.csv', index=False)

In [171]:
imars_crash_details.shape, imars_crash_slim_filtered.shape

((29373, 24), (24689, 12))

In [172]:
imars_crash_details_filtered = pd.merge(imars_crash_details, imars_crash_slim_filtered, how="left", on="IMARS_Record_No")
imars_crash_details_filtered.shape

(29373, 35)

In [173]:
imars_crash_details_filtered.to_csv('./VRP Needs Assessment/Crash Details Table.csv', index=False)

# Filter for Necessary Fields, Group by IMARS_RECORD_NO

## Passenger Table

In [99]:
imars_passenger = imars_passenger.rename(columns={'IMARS_Record_No':'INCID_NO'}) # rename to match CDS
imars_passenger_slim = imars_passenger[['INCID_NO']] # subset needed columns

In [100]:
imars_passenger['Injury_Severity'].value_counts()

01. No injury                    30127
02. Possible injury               4760
03. Non-incapacitating injury     2767
99. Unknown                       2585
04. Incapacitating injury         1008
05. Fatal                          268
Name: Injury_Severity, dtype: int64

In [101]:
# set flags for each injury severity 
imars_passenger_slim['No Injury']= np.where(imars_passenger['Injury_Severity']=='01. No injury', 1,0)
imars_passenger_slim['Possible Injury']= np.where(imars_passenger['Injury_Severity']=='02. Possible injury', 1,0)
imars_passenger_slim['Non-incapacitating Injury']= np.where(imars_passenger['Injury_Severity']=='03. Non-incapacitating injury', 1,0)
imars_passenger_slim['Incapacitating Injury']= np.where(imars_passenger['Injury_Severity']=='04. Incapacitating injury', 1,0)
imars_passenger_slim['Fatality']= np.where(imars_passenger['Injury_Severity']=='05. Fatal', 1,0)
imars_passenger_slim['Fatality'].sum() # exactly as expected from injury severity value counts

C:\Users\Sophie.Kaye\AppData\Local\Temp\ipykernel_47612\189481374.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  imars_passenger_slim['No Injury']= np.where(imars_passenger['Injury_Severity']=='01. No injury', 1,0)
C:\Users\Sophie.Kaye\AppData\Local\Temp\ipykernel_47612\189481374.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  imars_passenger_slim['Possible Injury']= np.where(imars_passenger['Injury_Severity']=='02. Possible injury', 1,0)
C:\Users\Sophie.Kaye\AppData\Local\Temp\ipykernel_47612\1894

268

In [102]:
imars_passenger['Pedestrian_Type'].value_counts()

Other         233
Bicyclist      71
Pedestrian     56
Name: Pedestrian_Type, dtype: int64

In [104]:
# determine the number of motorists and non-motorists involved
imars_passenger_slim['Num_Motorist']= np.where(imars_passenger['Pedestrian_Type'].isnull()==True, 1,0)
imars_passenger_slim['Num_Non_Motorist']= np.where(imars_passenger['Pedestrian_Type'].isnull()==False, 1,0)
# check new non-motorist logic, looks good!
imars_passenger_slim['Num_Non_Motorist'].sum(), (233+71+56) # combined number of Other, Bicyclists, and Pedestrians

C:\Users\Sophie.Kaye\AppData\Local\Temp\ipykernel_47612\4128283654.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  imars_passenger_slim['Num_Motorist']= np.where(imars_passenger['Pedestrian_Type'].isnull()==True, 1,0)
C:\Users\Sophie.Kaye\AppData\Local\Temp\ipykernel_47612\4128283654.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  imars_passenger_slim['Num_Non_Motorist']= np.where(imars_passenger['Pedestrian_Type'].isnull()==False, 1,0)


(360, 360)

In [105]:
imars_passenger_slim['Num_Pedestrian'] = np.where(imars_passenger['Pedestrian_Type']=='Pedestrian', 1,0)
imars_passenger_slim['Num_Pedestrian'].sum() # exactly as expected based on Pedestrian Type value counts

C:\Users\Sophie.Kaye\AppData\Local\Temp\ipykernel_47612\4013749937.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  imars_passenger_slim['Num_Pedestrian'] = np.where(imars_passenger['Pedestrian_Type']=='Pedestrian', 1,0)


56

In [106]:
imars_passenger_slim['Num_Ped_Deaths'] = np.where(imars_passenger_slim['Num_Pedestrian']==1, imars_passenger_slim['Fatality'], 0)
# check logic, looks good!
test = imars_passenger.loc[imars_passenger['Pedestrian_Type']=='Pedestrian']
imars_passenger_slim['Num_Ped_Deaths'].sum(), test[test.Injury_Severity=='05. Fatal'].shape[0] # 1 fatality in subset of passenger table for pedestrians only

C:\Users\Sophie.Kaye\AppData\Local\Temp\ipykernel_47612\1150133979.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  imars_passenger_slim['Num_Ped_Deaths'] = np.where(imars_passenger_slim['Num_Pedestrian']==1, imars_passenger_slim['Fatality'], 0)


(1, 1)

In [107]:
imars_passenger_slim.head()

,INCID_NO,No Injury,Possible Injury,Non-incapacitating Injury,Incapacitating Injury,Fatality,Num_Motorist,Num_Non_Motorist,Num_Pedestrian,Num_Ped_Deaths
0,PP24033388,1,0,0,0,0,1,0,0,0
1,PP24036230,1,0,0,0,0,1,0,0,0
2,PP24036442,1,0,0,0,0,1,0,0,0
3,PP24036614,1,0,0,0,0,1,0,0,0
4,PP24037029,1,0,0,0,0,1,0,0,0


In [108]:
imars_passenger_slim.shape

(54594, 10)

In [109]:
imars_passenger_slim.columns

Index(['INCID_NO', 'No Injury', 'Possible Injury', 'Non-incapacitating Injury',
       'Incapacitating Injury', 'Fatality', 'Num_Motorist', 'Num_Non_Motorist',
       'Num_Pedestrian', 'Num_Ped_Deaths'],
      dtype='object')

In [110]:
# collapse multiple rows for each person involved in the crash into a single row for each crash
# injuries/fatalities and number of people involved should be summed as total numbers per crash ("sum" function)
imars_passenger_slim_agg = imars_passenger_slim.groupby(by=['INCID_NO']).sum().reset_index()
imars_passenger_slim_agg.shape

(29373, 10)

In [111]:
imars_passenger_slim_agg['Num_Pedestrian'].value_counts()

0    29331
1       34
2        4
4        2
3        2
Name: Num_Pedestrian, dtype: int64

In [113]:
# set binary indicators for VRU and Pedestrian
imars_passenger_slim_agg['Pedestrian'] = np.where(imars_passenger_slim_agg['Num_Pedestrian'] > 0, 1, 0)
imars_passenger_slim_agg['VRU'] = np.where(imars_passenger_slim_agg['Num_Non_Motorist'] > 0, 1, 0)
# test logic, looks good!
imars_passenger_slim_agg['Pedestrian'].sum(), (34+4+2+2) # replacing number of pedestrians involved in each crash with 1 for all crashes involving pedestrians

(42, 42)

In [114]:
imars_passenger_slim_agg['Crash_Severity'] = 0 # initialize column with dummy data to replace with for loop
length = range(imars_passenger_slim_agg.shape[0]) # search through all rows in aggregated dataset (to examine all injuries from each crash as a whole)
for i in length: # determine most severe injury outcome
    if imars_passenger_slim_agg['Fatality'].iloc[i] > 0:
        imars_passenger_slim_agg['Crash_Severity'].iloc[i] = 'Fatal'
    elif imars_passenger_slim_agg['Incapacitating Injury'].iloc[i] > 0:
        imars_passenger_slim_agg['Crash_Severity'].iloc[i] = 'Incap'
    elif imars_passenger_slim_agg['Non-incapacitating Injury'].iloc[i] > 0:
        imars_passenger_slim_agg['Crash_Severity'].iloc[i] = 'Non-Incap'
    elif imars_passenger_slim_agg['Possible Injury'].iloc[i] > 0:
        imars_passenger_slim_agg['Crash_Severity'].iloc[i] = 'Possible'
    else:
        imars_passenger_slim_agg['Crash_Severity'].iloc[i] = 'No Inj'

C:\Users\Sophie.Kaye\AppData\Local\Temp\ipykernel_47612\775537522.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  imars_passenger_slim_agg['Crash_Severity'].iloc[i] = 'No Inj'


In [115]:
#set flags for each Crash Severity attribute
imars_passenger_slim_agg['CrashSeverity_Fatal'] = np.where(imars_passenger_slim_agg['Crash_Severity']=='Fatal', 1,0)
imars_passenger_slim_agg['CrashSeverity_Incap'] = np.where(imars_passenger_slim_agg['Crash_Severity']=='Incap', 1,0)
imars_passenger_slim_agg['CrashSeverity_NonIncap'] = np.where(imars_passenger_slim_agg['Crash_Severity']=='Non-Incap', 1,0)
imars_passenger_slim_agg['CrashSeverity_Possible'] = np.where(imars_passenger_slim_agg['Crash_Severity']=='Possible', 1,0)
imars_passenger_slim_agg['CrashSeverity_NoInj'] = np.where(imars_passenger_slim_agg['Crash_Severity']=='No Inj', 1,0)

In [116]:
# test crash severity logic - looks good!
test=imars_passenger_slim_agg.loc[imars_passenger_slim_agg['Fatality']==0] # remove fatal crashes
imars_passenger_slim_agg['CrashSeverity_Incap'].sum(), test.loc[test['Incapacitating Injury']>0].shape[0] # number of crashes with incapacitating severity = number of crashes resulting in incapacitating injuries when fatal crashes are removed from dataset

(787, 787)

In [117]:
imars_passenger_slim_agg.head()

,INCID_NO,No Injury,Possible Injury,Non-incapacitating Injury,Incapacitating Injury,Fatality,Num_Motorist,Num_Non_Motorist,Num_Pedestrian,Num_Ped_Deaths,Pedestrian,VRU,Crash_Severity,CrashSeverity_Fatal,CrashSeverity_Incap,CrashSeverity_NonIncap,CrashSeverity_Possible,CrashSeverity_NoInj
0,NP12000078,0,0,0,0,0,1,0,0,0,0,0,No Inj,0,0,0,0,1
1,NP12000378,0,0,0,0,0,1,0,0,0,0,0,No Inj,0,0,0,0,1
2,NP12000550,0,0,0,0,0,1,0,0,0,0,0,No Inj,0,0,0,0,1
3,NP12000911,0,0,0,0,0,1,0,0,0,0,0,No Inj,0,0,0,0,1
4,NP12000935,0,0,0,0,0,1,0,0,0,0,0,No Inj,0,0,0,0,1


In [118]:
imars_passenger_slim_agg.shape

(29373, 18)

In [119]:
imars_passenger_slim_agg.columns

Index(['INCID_NO', 'No Injury', 'Possible Injury', 'Non-incapacitating Injury',
       'Incapacitating Injury', 'Fatality', 'Num_Motorist', 'Num_Non_Motorist',
       'Num_Pedestrian', 'Num_Ped_Deaths', 'Pedestrian', 'VRU',
       'Crash_Severity', 'CrashSeverity_Fatal', 'CrashSeverity_Incap',
       'CrashSeverity_NonIncap', 'CrashSeverity_Possible',
       'CrashSeverity_NoInj'],
      dtype='object')

In [120]:
imars_passenger_slim_agg.to_csv("./IMARS/IMARS2024_passenger_slim_agg.csv",index=False)

## Vehicle Table

In [121]:
imars_vehicle['Posted_Speed'].value_counts()

09. 45 mph        6866
07. 35 mph        5880
11. 55 mph        5346
05. 25 mph        4959
10. 50 mph        3369
98. Not posted    2591
99. Unknown       2146
03. 15 mph        1792
08. 40 mph        1719
06. 30 mph         960
02. 10 mph         626
01. 5 mph          414
04. 20 mph         390
13. 65 mph          72
12. 60 mph          52
14. 70 mph           9
15. 75 mph           4
Name: Posted_Speed, dtype: int64

In [122]:
imars_vehicle['Traffic_Control'].value_counts().sort_index()

01. None                                      25518
02. Stop sign                                  1731
03. Yield sign                                  543
04. Flashing traffic signal                     103
05. Traffic signal                             2081
06. Pedestrian crossing                         292
07. Person (ofc/flagger, xing guard, etc.)      134
08. No passing zone                            2047
09. Warning signs                               379
10. Temporary traffic control                   288
12. RR crossing device/signal                     7
13. Other                                      1013
99. Unknown                                    1603
Name: Traffic_Control, dtype: int64

In [124]:
# number of vehicle entries in which a crash occurred at a known site other than a crosswalk (or person)
25518+1731+543+103+2081+2047+379+288+7+1013

33710

In [123]:
imars_vehicle = imars_vehicle.rename(columns={'IMARS_Record_No':'INCID_NO'}) # rename to match CDS
imars_vehicle_slim = imars_vehicle[['INCID_NO']] # subset needed columns
# set flags for speed limit
imars_vehicle_slim['5_mph']= np.where(imars_vehicle['Posted_Speed']=='01. 5 mph', 1,0)
imars_vehicle_slim['10_mph']= np.where(imars_vehicle['Posted_Speed']=='02. 10 mph', 1,0)
imars_vehicle_slim['15_mph']= np.where(imars_vehicle['Posted_Speed']=='03. 15 mph', 1,0)
imars_vehicle_slim['20_mph']= np.where(imars_vehicle['Posted_Speed']=='04. 20 mph', 1,0)
imars_vehicle_slim['25_mph']= np.where(imars_vehicle['Posted_Speed']=='05. 25 mph', 1,0)
imars_vehicle_slim['30_mph']= np.where(imars_vehicle['Posted_Speed']=='06. 30 mph', 1,0)
imars_vehicle_slim['35_mph']= np.where(imars_vehicle['Posted_Speed']=='07. 35 mph', 1,0)
imars_vehicle_slim['40_mph']= np.where(imars_vehicle['Posted_Speed']=='08. 40 mph', 1,0)
imars_vehicle_slim['45_mph']= np.where(imars_vehicle['Posted_Speed']=='09. 45 mph', 1,0)
imars_vehicle_slim['50_mph']= np.where(imars_vehicle['Posted_Speed']=='10. 50 mph', 1,0)
imars_vehicle_slim['55_mph']= np.where(imars_vehicle['Posted_Speed']=='11. 55 mph', 1,0)
imars_vehicle_slim['60_mph']= np.where(imars_vehicle['Posted_Speed']=='12. 60 mph', 1,0)
imars_vehicle_slim['65_mph']= np.where(imars_vehicle['Posted_Speed']=='13. 65 mph', 1,0)
imars_vehicle_slim['70_mph']= np.where(imars_vehicle['Posted_Speed']=='14. 70 mph', 1,0)
imars_vehicle_slim['75_mph']= np.where(imars_vehicle['Posted_Speed']=='15. 75 mph', 1,0)
imars_vehicle_slim['no_posted_speed']= np.where(imars_vehicle['Posted_Speed']=='98. Not posted', 1,0)
imars_vehicle_slim['Crosswalk']= np.where(imars_vehicle['Traffic_Control']=='06. Pedestrian crossing', 1,0)
imars_vehicle_slim['Outside a Crosswalk']= np.where(imars_vehicle['Traffic_Control'].isin(['01. None','02. Stop sign','03. Yield sign','04. Flashing traffic signal','05. Traffic signal','08. No passing zone','09. Warning signs','10. Temporary traffic control','12. RR crossing device/signal','13. Other']), 1,0)
imars_vehicle_slim['Outside a Crosswalk'].sum() # exactly as expected based on Traffic Control value counts

C:\Users\Sophie.Kaye\AppData\Local\Temp\ipykernel_47612\323004474.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  imars_vehicle_slim['5_mph']= np.where(imars_vehicle['Posted_Speed']=='01. 5 mph', 1,0)
C:\Users\Sophie.Kaye\AppData\Local\Temp\ipykernel_47612\323004474.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  imars_vehicle_slim['10_mph']= np.where(imars_vehicle['Posted_Speed']=='02. 10 mph', 1,0)
C:\Users\Sophie.Kaye\AppData\Local\Temp\ipykernel_47612\323004474.py:6: SettingWithCopyWarning: 
A v

33710

In [125]:
imars_vehicle_slim.columns

Index(['INCID_NO', '5_mph', '10_mph', '15_mph', '20_mph', '25_mph', '30_mph',
       '35_mph', '40_mph', '45_mph', '50_mph', '55_mph', '60_mph', '65_mph',
       '70_mph', '75_mph', 'no_posted_speed', 'Crosswalk',
       'Outside a Crosswalk'],
      dtype='object')

In [126]:
# collapse multiple rows for each vehicle involved in the crash into a single row for each crash
# only one speed limit should exist per crash to not double count data attribute ("first" function)
imars_vehicle_slim_agg = imars_vehicle_slim.groupby(by=['INCID_NO']).first().reset_index()
imars_vehicle_slim_agg.shape

(29373, 19)

In [127]:
imars_vehicle['Vehicle_Type'].value_counts()

01. Passenger car (sedan, minivan, etc.)         17980
02. Sport utility vehicle                         8894
05. Pickup truck                                  4793
10. Motorcycle > 150cc                            1485
06. Motor home                                     851
99. Unknown                                        775
18. Medium/heavy trucks (10,000 lbs. or more)      636
03. Passenger van (9-15 passengers)                428
17. Other light trucks (10,000 lbs. or less)       425
21. Other                                          406
04. Cargo van (10,000 lbs. or less)                384
08. Transit/shuttle bus                            373
19. Maintenance/construction vehicle               228
09. Motor coach                                    201
16. Snowmobile                                      88
15. ATV                                             73
07. School bus                                      64
11. Motorcycle < 150cc (trail bike)                 50
12. Moped/

In [128]:
imars_vehicle['NUM_VEH'] = 1 # one vehicle per entry
imars_vehicletypes = imars_vehicle[['INCID_NO', 'NUM_VEH']] # subset needed columns

In [129]:
# set flags for each vehicle type
imars_vehicletypes['Num_Motorcycle'] = np.where(imars_vehicle['Vehicle_Type']=='10. Motorcycle > 150cc', 1,0)
imars_vehicletypes['Pax'] = np.where(imars_vehicle['Vehicle_Type']=='01. Passenger car (sedan, minivan, etc.)', 1,0)
imars_vehicletypes['SUV'] = np.where(imars_vehicle['Vehicle_Type']=='02. Sport utility vehicle', 1,0)
imars_vehicletypes['Van'] = np.where(imars_vehicle['Vehicle_Type'].isin(['03. Passenger van (9-15 passengers)','04. Cargo van (10,000 lbs. or less)','05. Pickup truck'])==True, 1,0)
imars_vehicletypes['Bus'] = np.where(imars_vehicle['Vehicle_Type'].isin(['06. Motor home','07. School bus','08. Transit/shuttle bus','09. Motor coach'])==True, 1,0)
imars_vehicletypes['Medium Truck'] = np.where(imars_vehicle['Vehicle_Type']=='17. Other light trucks (10,000 lbs. or less)', 1,0)
imars_vehicletypes['Heavy Truck'] = np.where(imars_vehicle['Vehicle_Type']=='18. Medium/heavy trucks (10,000 lbs. or more)', 1,0)
imars_vehicletypes['Equipment'] = np.where(imars_vehicle['Vehicle_Type'].isin(['19. Maintenance/construction vehicle','20. Farm equipment'])==True, 1,0)
imars_vehicletypes['Nonroad HighSpeed'] = np.where(imars_vehicle['Vehicle_Type'].isin(['11. Motorcycle < 150cc (trail bike)','15. ATV','16. Snowmobile'])==True, 1,0)
imars_vehicletypes['Nonroad LowSpeed'] = np.where(imars_vehicle['Vehicle_Type'].isin(['12. Moped/minibike/scooter/segway','14. Low speed vehicle (golf cart, etc.)'])==True, 1,0)
imars_vehicletypes['Unknown99'] = np.where(imars_vehicle['Vehicle_Type']=='99. Unknown', 1,0)
imars_vehicletypes['Empty'] = np.where(pd.isna(imars_vehicle['Vehicle_Type']),1,0)
imars_vehicletypes['Unknown'] = imars_vehicletypes['Unknown99']+imars_vehicletypes['Empty']

C:\Users\Sophie.Kaye\AppData\Local\Temp\ipykernel_47612\4276648447.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  imars_vehicletypes['Num_Motorcycle'] = np.where(imars_vehicle['Vehicle_Type']=='10. Motorcycle > 150cc', 1,0)
C:\Users\Sophie.Kaye\AppData\Local\Temp\ipykernel_47612\4276648447.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  imars_vehicletypes['Pax'] = np.where(imars_vehicle['Vehicle_Type']=='01. Passenger car (sedan, minivan, etc.)', 1,0)
C:\Users\Sophie.Kaye\AppData\Local\Temp\ipykern

In [130]:
imars_vehicletypes = imars_vehicletypes.drop(columns={'Unknown99', 'Empty'}) # drop columns used for calculated fields
# collapse multiple rows for each vehicle involved in the crash into a single row for each crash
# vehicle types and number of people involved should be summed as total numbers per crash ("sum" function)
imars_vehicletypes_agg = imars_vehicletypes.groupby(by=['INCID_NO']).sum().reset_index()
imars_vehicletypes_agg.shape

(29373, 13)

In [131]:
imars_vehicletypes_agg.head()

,INCID_NO,NUM_VEH,Num_Motorcycle,Pax,SUV,Van,Bus,Medium Truck,Heavy Truck,Equipment,Nonroad HighSpeed,Nonroad LowSpeed,Unknown
0,NP12000078,1,0,0,0,0,0,0,0,0,0,0,1
1,NP12000378,1,0,0,0,0,0,0,0,0,0,0,1
2,NP12000550,1,0,0,0,0,0,0,0,0,0,0,1
3,NP12000911,1,0,0,0,0,0,0,0,0,0,0,1
4,NP12000935,1,0,0,0,0,0,0,0,0,0,0,1


In [132]:
imars_vehicletypes_agg['Num_Motorcycle'].value_counts()

0    28025
1     1239
2       96
4        5
3        5
6        2
7        1
Name: Num_Motorcycle, dtype: int64

In [133]:
# set indicator for crashes involving motorcycles
imars_vehicletypes_agg['Motorcycle_Ind'] = np.where(imars_vehicletypes_agg['Num_Motorcycle'] > 0, 1,0)
imars_vehicletypes_agg['Motorcycle_Ind'].sum(), (1239+96+5+5+2+1) #  looks good (replacing number of motorcyclists involved in each crash with 1 for all crashes involving motorcyclists)

(1348, 1348)

In [134]:
imars_vehicletypes_agg.columns

Index(['INCID_NO', 'NUM_VEH', 'Num_Motorcycle', 'Pax', 'SUV', 'Van', 'Bus',
       'Medium Truck', 'Heavy Truck', 'Equipment', 'Nonroad HighSpeed',
       'Nonroad LowSpeed', 'Unknown', 'Motorcycle_Ind'],
      dtype='object')

In [135]:
# join aggregated ('sum' and 'first') vehicle data into single table
imars_slim_vehicle_agg = imars_vehicle_slim_agg.merge(imars_vehicletypes_agg, how='right', on='INCID_NO')
imars_slim_vehicle_agg = imars_slim_vehicle_agg.drop_duplicates()
imars_slim_vehicle_agg.shape

(29373, 32)

In [136]:
imars_slim_vehicle_agg.columns

Index(['INCID_NO', '5_mph', '10_mph', '15_mph', '20_mph', '25_mph', '30_mph',
       '35_mph', '40_mph', '45_mph', '50_mph', '55_mph', '60_mph', '65_mph',
       '70_mph', '75_mph', 'no_posted_speed', 'Crosswalk',
       'Outside a Crosswalk', 'NUM_VEH', 'Num_Motorcycle', 'Pax', 'SUV', 'Van',
       'Bus', 'Medium Truck', 'Heavy Truck', 'Equipment', 'Nonroad HighSpeed',
       'Nonroad LowSpeed', 'Unknown', 'Motorcycle_Ind'],
      dtype='object')

In [137]:
imars_slim_vehicle_agg.to_csv("./IMARS/IMARS2024_slim_vehicle_agg.csv",index=False)

## Crash Details Table

In [138]:
imars_crash_details = imars_crash_details.rename(columns={'IMARS_Record_No':'INCID_NO'}) # rename to match CDS
imars_crash_details_slim = imars_crash_details[['INCID_NO', 'First_Harmful_Event', 'First_Harmful_Event_Type']] # subset needed columns

In [139]:
pd.options.display.max_rows = 10000000
imars_crash_details['First_Harmful_Event'].value_counts()

21. Motor vehicle in transport                                           11064
22. Parked motor vehicle                                                  2501
52. Tree/shrub                                                            2093
28. Deer                                                                  1522
07. Ran off the road right                                                1399
99. Unknown                                                                621
66. Other fixed object                                                     592
25. Other non-fixed object                                                 542
60. Rock, boulder, rock slide                                              531
46. Earth embankment/berm                                                  519
16. Other non-collision                                                    518
01. Overturn/rollover                                                      450
08. Ran Off the road left                           

In [140]:
imars_crash_details['First_Harmful_Event_Type'].value_counts()

Collision with person, MV or non-fixed object    14743
Collision with fixed object                       7112
Non-collision                                     3396
Collision with animals                            2243
Unknown                                            627
Name: First_Harmful_Event_Type, dtype: int64

In [141]:
imars_crash_details['Environmental_Contributing_Circumstances'].value_counts()

01. None                                                                                    20180
02. Weather                                                                                  2398
06. Animal(s) in roadway                                                                     1655
99. Unknown                                                                                  1370
03. Physical obstruction(s) (trees, bushes, etc.)                                             449
07. Other                                                                                     415
05. Glare                                                                                     162
02. Weather; 03. Physical obstruction(s) (trees, bushes, etc.)                                154
02. Weather; 06. Animal(s) in roadway                                                          86
02. Weather; 07. Other                                                                         82
04. Rockfall        

In [143]:
# set flags for animal related data attributes
imars_crash_details_slim['Collision w Animal']= np.where(imars_crash_details['First_Harmful_Event_Type']=="Collision with animals", 1,0)
imars_crash_details_slim['Avoiding Animal']= np.where(imars_crash_details['First_Harmful_Event']=="13. Avoiding an animal on road", 1,0)
imars_crash_details_slim['Animal in Roadway']= np.where(imars_crash_details['Environmental_Contributing_Circumstances'].str.contains("Animal")==True, 1,0)
# check logic - looks good!
imars_crash_details_slim['Animal in Roadway'].sum(), (1655+86+23+18+17+8+7+3+3+2+1+1+1+1) # includes all possible animal-related entries in environmental value counts

C:\Users\Sophie.Kaye\AppData\Local\Temp\ipykernel_47612\494197613.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  imars_crash_details_slim['Collision w Animal']= np.where(imars_crash_details['First_Harmful_Event_Type']=="Collision with animals", 1,0)
C:\Users\Sophie.Kaye\AppData\Local\Temp\ipykernel_47612\494197613.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  imars_crash_details_slim['Avoiding Animal']= np.where(imars_crash_details['First_Harmful_Event']=="13. Avoiding an animal on road", 1,0)
C:

(1826, 1826)

In [144]:
# set animal indicator
animal = imars_crash_details_slim[['Collision w Animal', 'Animal in Roadway', 'Avoiding Animal']]
imars_crash_details_slim['Involving Animal'] = [1 if x > 0 else 0 for x in np.sum(animal.values == 1, 1)] # assign value of 1 in new "involving animal" column if flags are set for at least one animal indicator 
imars_crash_details_slim['Involving Animal'].sum() # greater value than either Collision w Animal, Animal in Roadway, or Avoiding Animal so it must be accounting for them all

C:\Users\Sophie.Kaye\AppData\Local\Temp\ipykernel_47612\3562255218.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  imars_crash_details_slim['Involving Animal'] = [1 if x > 0 else 0 for x in np.sum(animal.values == 1, 1)] # assign value of 1 in new "involving animal" column if flags are set for at least one animal indicator


2729

In [145]:
# set flags for lighting characteristics
imars_crash_details_slim['Daylight'] = np.where(imars_crash_details['Lighting']=="01. Daylight", 1,0)
imars_crash_details_slim['Dawn'] = np.where(imars_crash_details['Lighting']=="02. Dawn", 1,0)
imars_crash_details_slim['Dusk'] = np.where(imars_crash_details['Lighting']=="03. Dusk", 1,0)
imars_crash_details_slim['Dark_Lit'] = np.where(imars_crash_details['Lighting']=="04. Dark - lighted", 1,0)
imars_crash_details_slim['Dark_NotLit'] = np.where(imars_crash_details['Lighting']=="05. Dark - not lighted", 1,0)
imars_crash_details_slim['Dark_UnknownLit'] = np.where(imars_crash_details['Lighting']=="06. Dark - unknown lighting", 1,0)
imars_crash_details_slim['UnknownLit'] = np.where(imars_crash_details['Lighting']=="99. Unknown", 1,0)
imars_crash_details_slim['NullLit'] = np.where(pd.isna(imars_crash_details['Lighting']),1,0)
imars_crash_details_slim['UnknownBlankNullLit'] = imars_crash_details_slim['UnknownLit']+imars_crash_details_slim['NullLit'] # combined field for which unspecified lighting attribute may be able to be assigned based on sun location with respect to time
# check sum
imars_crash_details_slim['Daylight'].sum(), imars_crash_details_slim['Dawn'].sum(), imars_crash_details_slim['Dusk'].sum(), imars_crash_details_slim['Dark_Lit'].sum(), imars_crash_details_slim['Dark_NotLit'].sum(), imars_crash_details_slim['Dark_UnknownLit'].sum(), imars_crash_details_slim['UnknownLit'].sum(), imars_crash_details_slim['NullLit'].sum(), imars_crash_details_slim['UnknownBlankNullLit'].sum()

C:\Users\Sophie.Kaye\AppData\Local\Temp\ipykernel_47612\694506141.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  imars_crash_details_slim['Daylight'] = np.where(imars_crash_details['Lighting']=="01. Daylight", 1,0)
C:\Users\Sophie.Kaye\AppData\Local\Temp\ipykernel_47612\694506141.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  imars_crash_details_slim['Dawn'] = np.where(imars_crash_details['Lighting']=="02. Dawn", 1,0)
C:\Users\Sophie.Kaye\AppData\Local\Temp\ipykernel_47612\694506141.py:4: SettingW

(19242, 641, 961, 1246, 4766, 277, 1172, 964, 2136)

In [146]:
imars_crash_details['Lighting'].value_counts()

01. Daylight                   19242
05. Dark - not lighted          4766
04. Dark - lighted              1246
99. Unknown                     1172
03. Dusk                         961
02. Dawn                         641
06. Dark - unknown lighting      277
07. Other                        104
Name: Lighting, dtype: int64

In [147]:
imars_crash_details.loc[imars_crash_details['Lighting'].isnull()].shape[0]

964

In [148]:
imars_crash_details.shape[0], imars_crash_expanded.shape[0] # fewer records in "crash_expanded" table, therefore following join should be "right" rather than "left"

(29373, 26518)

In [149]:
# add region and time columns from IMARS crash table to IMARS crash details table to filter lighting attributes based on region, joining the two datasets based on INCID_NO
imars_crash_details = pd.merge(imars_crash_details, imars_crash_expanded[['RGN','TIME','INCID_NO']], how='right', on='INCID_NO')
imars_crash_details = imars_crash_details.drop_duplicates() 
imars_crash_details.head()

,INCID_NO,Crash_Date_Time,Number_of_Vehicles_Involved,Injury_or_Fatal_Crash,Investigated_at_Scene,Hit_and_Run,Non_Motor_Vehicl_Property_Damage,Amount_of_Property_Damage,First_Harmful_Event_Type,First_Harmful_Event,...,Environmental_Contributing_Circumstances,Work_Zone_Related,Work_Zone_Workers_Present,Work_Zone_Location,Law_Enforcement_Present_at_Work_Zone,Relation_to_Junction,Type_of_Intersection,Manner_of_Collision,RGN,TIME
0,PP17080295,20180803 09:44:00:000,1.0,01. Yes,Yes,No,Yes,Unknown,Collision with fixed object,49. Guardrail/concrete traffic barrier,...,01. None,02. No,02. No,01. Not applicable,01. No,01. Non-junction,01. Not an intersection,"04. Sideswipe, same direction",NCR,09:44
1,PP18027209,20180401 15:43:00:000,2.0,02. No,Yes,Yes,Unknown,Unknown,Unknown,99. Unknown,...,01. None,02. No,NaN,NaN,NaN,01. Non-junction,01. Not an intersection,01. Front to rear,NCR,15:43
2,PP18027231,20180401 16:39:00:000,2.0,01. Yes,Yes,No,No,Unknown,"Collision with person, MV or non-fixed object",21. Motor vehicle in transport,...,01. None,02. No,02. No,01. Not applicable,01. No,02. Intersection,03. T intersection,08. Other,NCR,16:39
3,PP14068901,20210902 15:12:00:000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NCR,15:12
4,PP18027070,20180401 04:30:00:000,2.0,02. No,No,Yes,No,Over $1000,"Collision with person, MV or non-fixed object",21. Motor vehicle in transport,...,01. None,02. No,02. No,01. Not applicable,01. No,01. Non-junction,01. Not an intersection,01. Front to rear,NCR,04:30


In [150]:
# replace unspecified null Lighting attributes with more accurate data based on general sun position with respect to time
# times from NOAA: https://vividmaps.com/sunrise-time-sunset-time-and-hours-of/ and https://www.google.com/search?sca_esv=572573644&q=sunrise+and+sunset+on+winter+solstice+map&tbm=isch&source=lnms&sa=X&ved=2ahUKEwjgnaaU9O6BAxUlFFkFHeLJDn8Q0pQJegQIDRAB&biw=1912&bih=966&dpr=1

length = range(imars_crash_details.shape[0]) # search through all rows in dataset
for i in length: # assign lighting condition based on crash hour when unknown
    if imars_crash_details["RGN"].iloc[i] == "NCR": # NCR has more precise guaranteed lighting times due to concentrated crash locations
        if imars_crash_details_slim['UnknownBlankNullLit'].iloc[i] == 1:
            if imars_crash_details['TIME'].iloc[i] > '08:29' and imars_crash_details['TIME'].iloc[i] < '15:31': # guaranteed daylight period w/ hour buffer
                imars_crash_details_slim['Daylight'].iloc[i] = 1
                imars_crash_details_slim['UnknownBlankNullLit'].iloc[i] = 0
            elif imars_crash_details['TIME'].iloc[i] > '21:59' or imars_crash_details['TIME'].iloc[i] < '04:31': # guaranteed dark period w/ hour buffer 
                imars_crash_details_slim['Dark_UnknownLit'].iloc[i] = 1
                imars_crash_details_slim['UnknownBlankNullLit'].iloc[i] = 0
    else: # all other regions with less precise guaranteed lighting times due to spread crash locations
        if imars_crash_details_slim['UnknownBlankNullLit'].iloc[i] == 1:
            if imars_crash_details['TIME'].iloc[i] > '09:59' and imars_crash_details['TIME'].iloc[i] < '14:31':
                imars_crash_details_slim['Daylight'].iloc[i] = 1
                imars_crash_details_slim['UnknownBlankNullLit'].iloc[i] = 0
            elif imars_crash_details['TIME'].iloc[i] > '22:59' or imars_crash_details['TIME'].iloc[i] < '03:31':
                imars_crash_details_slim['Dark_UnknownLit'].iloc[i] = 1 
                imars_crash_details_slim['UnknownBlankNullLit'].iloc[i] = 0

# check sum
imars_crash_details_slim['Daylight'].sum(), imars_crash_details_slim['Dawn'].sum(), imars_crash_details_slim['Dusk'].sum(), imars_crash_details_slim['Dark_Lit'].sum(), imars_crash_details_slim['Dark_NotLit'].sum(), imars_crash_details_slim['Dark_UnknownLit'].sum(), imars_crash_details_slim['UnknownBlankNullLit'].sum()

C:\Users\Sophie.Kaye\AppData\Local\Temp\ipykernel_47612\2232653296.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  imars_crash_details_slim['Daylight'].iloc[i] = 1
C:\Users\Sophie.Kaye\AppData\Local\Temp\ipykernel_47612\2232653296.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  imars_crash_details_slim['UnknownBlankNullLit'].iloc[i] = 0
C:\Users\Sophie.Kaye\AppData\Local\Temp\ipykernel_47612\2232653296.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#retu

(19932, 641, 961, 1246, 4766, 424, 1299)

In [152]:
# group adverse lighting attributes into bin for poor visibility
PoorLighting = imars_crash_details_slim[['Dawn','Dusk','Dark_NotLit','Dark_Lit','Dark_UnknownLit']]
imars_crash_details_slim['Poor Lighting'] = [1 if x > 0 else 0 for x in np.sum(PoorLighting.values == 1, 1)] # assign value of 1 in new "poor lighting" column if flags are set for at least one darkness indicator 
imars_crash_details_slim['Poor Lighting'].sum(), (641+961+1246+4766+424) # check logic - looks good!

(8038, 8038)

In [153]:
#imars_crash_details['Weather'].value_counts()

In [154]:
#imars_crash_details['Roadway_Condition'].value_counts()

In [155]:
#imars_crash_details['AS_Road_Circumstance'].value_counts()

In [156]:
## set flags for adverse visibility conditions due to precip
#imars_crash_details_slim['Bad Weather'] = np.where(imars_crash_details['Weather'].str.contains('02|03|04|05|07|08|09|11')==True, 1,0)
#imars_crash_details_slim['Bad Road Condition'] = np.where(imars_crash_details['Roadway_Condition'].str.contains('02|03|04|05|06')==True, 1,0)
#imars_crash_details_slim['Bad Road Circumstance'] = np.where(imars_crash_details['AS_Road_Circumstance'].str.contains('02')==True, 1,0)
#imars_crash_details_slim['Bad Environmental Circumstance'] = np.where(imars_crash_details['Environmental_Contributing_Circumstances'].str.contains('02')==True, 1,0)

In [157]:
#imars_crash_details_slim['Bad Weather'].sum(), (847+195+103+18+18+15+11+10+2+256+78+54+45+11+6+4+3+1+87+10+6+3+3+1+7+2+1+4+2+1+20+8+5+1+12+5+1+1+18+11+3+1)

In [158]:
#imars_crash_details_slim['Bad Road Condition'].sum(), (1591+60+52+41+32+23+19+17+3+2+2+370+162+58+6+3+2+1+1+25+18+1+1+175+5+4+4+1+1+4)

In [159]:
#imars_crash_details_slim['Bad Road Circumstance'].sum(), (1503+75+43+42+38+19+17+15+11+10+7+6+5+5+5+5+3+3+3+3+2+2+2+2+2+2+2+2+1+1+1+1+1+1+1+1)

In [160]:
#imars_crash_details_slim['Bad Environmental Circumstance'].sum(), (1270+86+56+50+13+9+5+3+3+1+1+1+1+1+1)

In [161]:
## group adverse weather attributes into bin for poor visibility
#AdverseWeather = imars_crash_details_slim[['Bad Weather','Bad Road Condition','Bad Road Circumstance','Bad Environmental Circumstance']]
#imars_crash_details_slim['Adverse Weather'] = [1 if x > 0 else 0 for x in np.sum(AdverseWeather.values == 1, 1)] # assign value of 1 in new "adverse weather" column if flags are set for at least one bad weather indicator 
#imars_crash_details_slim['Adverse Weather'].sum() # greater value than either Bad Road Condition, Bad Weather, Bad Road Circumstance, or Bad Environmental Circumstance so it must be accounting for them all

In [162]:
imars_crash_details_slim.columns

Index(['INCID_NO', 'First_Harmful_Event', 'First_Harmful_Event_Type',
       'Collision w Animal', 'Avoiding Animal', 'Animal in Roadway',
       'Involving Animal', 'Daylight', 'Dawn', 'Dusk', 'Dark_Lit',
       'Dark_NotLit', 'Dark_UnknownLit', 'UnknownLit', 'NullLit',
       'UnknownBlankNullLit', 'Poor Lighting'],
      dtype='object')

In [163]:
imars_crash_details_slim = imars_crash_details_slim.drop(columns={'Collision w Animal', 'Avoiding Animal', 'Animal in Roadway', 'UnknownLit', 'NullLit'})#, 'Bad Weather', 'Bad Road Condition', 'Bad Road Circumstance', 'Bad Environmental Circumstance'}) # get rid of columns used to create indicators that will not be needed for analysis
imars_crash_details_slim.columns

Index(['INCID_NO', 'First_Harmful_Event', 'First_Harmful_Event_Type',
       'Involving Animal', 'Daylight', 'Dawn', 'Dusk', 'Dark_Lit',
       'Dark_NotLit', 'Dark_UnknownLit', 'UnknownBlankNullLit',
       'Poor Lighting'],
      dtype='object')

In [164]:
imars_crash_details_slim.shape

(29373, 12)

In [165]:
imars_crash_details_slim.to_csv("./IMARS/IMARS2024_crash_details_slim.csv",index=False)

# Merge Slim Tables

In [166]:
imars_crash_slim.shape, imars_crash_details_slim.shape, imars_passenger_slim_agg.shape, imars_slim_vehicle_agg.shape

((26518, 12), (29373, 12), (29373, 18), (29373, 32))

In [167]:
imars_crash_details_slim_merged = imars_crash_details_slim.merge(imars_crash_slim, how='right', on='INCID_NO')
imars_crash_details_slim_merged.shape

(26518, 23)

In [168]:
imars_crash_details_slim_merged.columns

Index(['INCID_NO', 'First_Harmful_Event', 'First_Harmful_Event_Type',
       'Involving Animal', 'Daylight', 'Dawn', 'Dusk', 'Dark_Lit',
       'Dark_NotLit', 'Dark_UnknownLit', 'UnknownBlankNullLit',
       'Poor Lighting', 'LATITUDE', 'LONGITUDE', 'Park', 'RGN', 'YEAR',
       'MONTH', 'DOW', 'HOUR', 'TIME', 'Urban', 'Linked_Address'],
      dtype='object')

In [169]:
imars_crash_details_and_passenger_slim = imars_passenger_slim_agg.merge(imars_crash_details_slim_merged, how='right', on='INCID_NO')
imars_crash_details_and_passenger_slim.shape

(26518, 40)

In [170]:
imars_crash_details_and_passenger_slim.columns

Index(['INCID_NO', 'No Injury', 'Possible Injury', 'Non-incapacitating Injury',
       'Incapacitating Injury', 'Fatality', 'Num_Motorist', 'Num_Non_Motorist',
       'Num_Pedestrian', 'Num_Ped_Deaths', 'Pedestrian', 'VRU',
       'Crash_Severity', 'CrashSeverity_Fatal', 'CrashSeverity_Incap',
       'CrashSeverity_NonIncap', 'CrashSeverity_Possible',
       'CrashSeverity_NoInj', 'First_Harmful_Event',
       'First_Harmful_Event_Type', 'Involving Animal', 'Daylight', 'Dawn',
       'Dusk', 'Dark_Lit', 'Dark_NotLit', 'Dark_UnknownLit',
       'UnknownBlankNullLit', 'Poor Lighting', 'LATITUDE', 'LONGITUDE', 'Park',
       'RGN', 'YEAR', 'MONTH', 'DOW', 'HOUR', 'TIME', 'Urban',
       'Linked_Address'],
      dtype='object')

In [171]:
imars_slim_all = imars_slim_vehicle_agg.merge(imars_crash_details_and_passenger_slim, how='right', on='INCID_NO')
imars_slim_all['database'] = 'IMARS' # add column with database name
imars_slim_all.shape

(26518, 72)

In [172]:
imars_slim_all.columns

Index(['INCID_NO', '5_mph', '10_mph', '15_mph', '20_mph', '25_mph', '30_mph',
       '35_mph', '40_mph', '45_mph', '50_mph', '55_mph', '60_mph', '65_mph',
       '70_mph', '75_mph', 'no_posted_speed', 'Crosswalk',
       'Outside a Crosswalk', 'NUM_VEH', 'Num_Motorcycle', 'Pax', 'SUV', 'Van',
       'Bus', 'Medium Truck', 'Heavy Truck', 'Equipment', 'Nonroad HighSpeed',
       'Nonroad LowSpeed', 'Unknown', 'Motorcycle_Ind', 'No Injury',
       'Possible Injury', 'Non-incapacitating Injury', 'Incapacitating Injury',
       'Fatality', 'Num_Motorist', 'Num_Non_Motorist', 'Num_Pedestrian',
       'Num_Ped_Deaths', 'Pedestrian', 'VRU', 'Crash_Severity',
       'CrashSeverity_Fatal', 'CrashSeverity_Incap', 'CrashSeverity_NonIncap',
       'CrashSeverity_Possible', 'CrashSeverity_NoInj', 'First_Harmful_Event',
       'First_Harmful_Event_Type', 'Involving Animal', 'Daylight', 'Dawn',
       'Dusk', 'Dark_Lit', 'Dark_NotLit', 'Dark_UnknownLit',
       'UnknownBlankNullLit', 'Poor Lighting', 

In [173]:
imars_slim_all.to_csv("./IMARS/IMARS2024_slim_all_clean.csv", index=False)

# Combine old and new IMARS data (already did this in the beginning of this notebook)

In [174]:
#imars_2023 = pd.read_csv("./IMARS_slim_all_clean.csv")
#imars_2022 = pd.read_csv("./IMARS_slim_clean_2022.csv") # to grab a few old columns
#imars_old = pd.merge(imars_2023, imars_2022[['INCID_NO','Crosswalk','Outside a Crosswalk']], on='INCID_NO')
#imars_old.shape

In [175]:
# reorder columns to match current IMARS
#imars_old = imars_old[['INCID_NO', '5_mph', '10_mph', '15_mph', '20_mph', '25_mph', '30_mph',
#       '35_mph', '40_mph', '45_mph', '50_mph', '55_mph', '60_mph', '65_mph',
#       '70_mph', '75_mph', 'no_posted_speed', 'Crosswalk',
#       'Outside a Crosswalk', 'NUM_VEH', 'Num_Motorcycle', 'Pax', 'SUV', 'Van',
#       'Bus', 'Medium Truck', 'Heavy Truck', 'Equipment', 'Nonroad HighSpeed',
#       'Nonroad LowSpeed', 'Unknown', 'Motorcycle_Ind', 'No Injury',
#       'Possible Injury', 'Non-incapacitating Injury', 'Incapacitating Injury',
#       'Fatality', 'Num_Motorist', 'Num_Non_Motorist', 'Num_Pedestrian',
#       'Num_Ped_Deaths', 'Pedestrian', 'VRU', 'Crash_Severity',
#       'CrashSeverity_Fatal', 'CrashSeverity_Incap', 'CrashSeverity_NonIncap',
#       'CrashSeverity_Possible', 'CrashSeverity_NoInj', 'First_Harmful_Event',
#       'First_Harmful_Event_Type', 'Involving Animal', 'Daylight', 'Dawn',
#       'Dusk', 'Dark_Lit', 'Dark_NotLit', 'Dark_UnknownLit',
#       'UnknownBlankNullLit', 'Poor Lighting', 'LATITUDE', 'LONGITUDE', 'Park',
#       'RGN', 'YEAR', 'MONTH', 'DOW', 'HOUR', 'TIME', 'Urban', 'database']]

In [176]:
# check if there is any overlapping reports between new and old IMARS data
#test = pd.merge(imars_slim_all, imars_old, how='inner', on='INCID_NO', suffixes=('_new','_old'))
#test.shape

In [177]:
# there are 2028 overlapping records, so which ones should take precedence? Have they been updated?
#test['YEAR_new'].value_counts()

In [178]:
#test['Fatality_new'].sum(), test['Fatality_old'].sum() # no fatality discrepancies between duplicate crash reports

In [179]:
#test['Incapacitating Injury_new'].sum(), test['Incapacitating Injury_old'].sum() # no serious injury discrepancies between duplicate crash reports

In [180]:
#test['LATITUDE_new'].isna().sum(), test['LATITUDE_old'].isna().sum() # more coordinates filled in for newer overlapping reports, so let's go with that

In [181]:
# drop rows from old database that are common with new database such that updated reports will prevail in final dataset with no overlap
#imars_old_nodupes = imars_old.loc[~imars_old['INCID_NO'].isin(imars_slim_all['INCID_NO'])] # Note that tilde symbol indicates NOT IN
#imars_old_nodupes.shape[0], (imars_old.shape[0] - test.shape[0])

In [182]:
#imars_slim_combinedyears = pd.concat([imars_slim_all, imars_old_nodupes])
#imars_slim_combinedyears.shape[0], (imars_slim_all.shape[0] + imars_old_nodupes.shape[0])

In [183]:
#imars_slim_combinedyears.to_csv("./IMARS/IMARS_slim_yearscombined_nodupes.csv", index=False)

# Output cleaned data for geolocation quality assessment

In [171]:
noLat = imars_slim_combinedyears.loc[imars_slim_combinedyears['LATITUDE'].isnull()]
noLong = imars_slim_combinedyears.loc[imars_slim_combinedyears['LONGITUDE'].isnull()]
# "no coords" must include all rows with at least one coordinate missing such that it cannot be geolocated
noLong_noDupes = noLong.loc[~noLong['INCID_NO'].isin(noLat['INCID_NO'])]
noCoords = pd.concat([noLat, noLong_noDupes])
noCoords_slim = noCoords[['Park','RGN','LATITUDE','LONGITUDE']]
noCoords_slim.to_csv('./IMARS/IMARS_crash_slim_NoCoords.csv', index=False)

In [169]:
lat = imars_slim_combinedyears.loc[imars_slim_combinedyears['LATITUDE'].notnull()]
coords = lat.loc[lat['LONGITUDE'].notnull()]
coords_slim = coords[['Park','RGN','LATITUDE','LONGITUDE']]
coords_slim.to_csv('./IMARS/IMARS_crash_slim_Coords.csv', index=False)